# Step 3: Customer Segmentation (RFM + Clustering)

**Objective:** Segment customers based on purchasing behavior.

**File:** `notebooks/3_customer_segmentation.ipynb`

## Actions:
1. Calculate RFM (Recency, Frequency, Monetary).
2. Visualize distributions.
3. Perform KMeans Clustering.
4. Visualize clusters.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

sns.set(style="whitegrid")

In [ ]:
# Load cleaned data
input_file = '../dataset/cleaned_data.pkl'
df = pd.read_pickle(input_file)

In [ ]:
# RFM Calculation
df_rfm = df[['customer_unique_id', 'order_purchase_timestamp', 'price']].copy()

max_date = df_rfm['order_purchase_timestamp'].max()
recency_df = df_rfm.groupby('customer_unique_id')['order_purchase_timestamp'].max().reset_index()
recency_df['Recency'] = (max_date - recency_df['order_purchase_timestamp']).dt.days

frequency_df = df_rfm.groupby('customer_unique_id')['order_purchase_timestamp'].count().reset_index()
frequency_df.columns = ['customer_unique_id', 'Frequency']

monetary_df = df_rfm.groupby('customer_unique_id')['price'].sum().reset_index()
monetary_df.columns = ['customer_unique_id', 'Monetary']

rfm = recency_df[['customer_unique_id', 'Recency']].merge(frequency_df, on='customer_unique_id')
rfm = rfm.merge(monetary_df, on='customer_unique_id')

In [ ]:
# RFM Visualizations
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.histplot(rfm['Recency'], bins=50)
plt.title('Recency Distribution')

plt.subplot(1, 3, 2)
sns.histplot(rfm['Frequency'], bins=50)
plt.title('Frequency Distribution')

plt.subplot(1, 3, 3)
sns.histplot(rfm['Monetary'], bins=50)
plt.title('Monetary Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Clustering
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

kmeans = KMeans(n_clusters=4, random_state=42)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

In [ ]:
# Cluster Visualization
plt.figure(figsize=(10, 6))
sns.scatterplot(data=rfm, x='Recency', y='Monetary', hue='Cluster', palette='viridis', alpha=0.6)
plt.title('Customer Segments (Recency vs Monetary)')
plt.show()